## Harmonic Langevin solver using sse_solver_py

The langevin equation can be solved for a harmonic system.

$$
\frac{dx}{dt} = \frac{p}{m}, \quad \frac{dp}{dt} = -m \omega^2 x - \gamma p + \sqrt{2 \gamma k_B T m} \xi(t),
$$


In [ ]:
import dataclasses

from slate_core import Array, FundamentalBasis, array, plot
from slate_core.metadata import (
    Domain,
)

from slate_quantum.dynamics.langevin import (
    LangevinParameters,
    solve_harmonic_langevin,
)
from slate_quantum.metadata import SpacedTimeMetadata

x_0 = 5.0
p_0 = 0
parameters = LangevinParameters(
    temperature=5.0, lambda_=0.01, mass=1.0, hbar=1.0, boltzmann=1.0
)
omega = 1.0
alpha_0 = parameters.eval_alpha(x_0, p_0)
times = FundamentalBasis(
    SpacedTimeMetadata(
        6000, domain=Domain(delta=300 / parameters.kbt_div_hbar), interpolation="DST"
    )
)
parameters = dataclasses.replace(parameters, temperature=0.01)
parameters = dataclasses.replace(parameters, lambda_=0.1)

alpha_points = solve_harmonic_langevin(alpha_0, times, parameters, omega)
alpha_points = array.as_fundamental_basis(alpha_points)
x_vals, p_vals = parameters.eval_xp(alpha_points.raw_data)
x_points = Array(alpha_points.basis, x_vals)
p_points = Array(alpha_points.basis, p_vals)

fig, ax, line_0 = plot.array_against_axes_1d(x_points, axes=(1,), measure="real")
ax.set_ylabel("Position / m")
ax.set_xlabel("Time / s")
ax.set_ylim(-10, 10)
(_, ax, line_1) = plot.array_against_axes_1d(
    p_points, axes=(1,), ax=ax.twinx(), measure="real"
)
ax.set_ylim(-10, 10)
line_1.set_color("C2")
ax.set_ylabel("Momentum / kgm/s")

ax.legend(handles=[line_0, line_1], labels=["Position", "Momentum"])
ax.set_title("Langevin Dynamics of Harmonic Oscillator")


We find that the open system exchanges energy with the bath and exists in dynamic equilibrium. However the closed harmonic oscillator oscillates indefinitely.


In [ ]:
parameters = dataclasses.replace(parameters, lambda_=0.0)
alpha_points = solve_harmonic_langevin(alpha_0, times, parameters, omega)

alpha_points = array.as_fundamental_basis(alpha_points)
x_vals, p_vals = parameters.eval_xp(alpha_points.raw_data)
x_points = Array(alpha_points.basis, x_vals)
p_points = Array(alpha_points.basis, p_vals)

fig, ax, line_0 = plot.array_against_axes_1d(x_points, axes=(1,), measure="real")
ax.set_ylabel("Position / m")
ax.set_xlabel("Time / s")
ax.set_ylim(-10, 10)
(_, ax, line_1) = plot.array_against_axes_1d(
    p_points, axes=(1,), ax=ax.twinx(), measure="real"
)
ax.set_ylim(-10, 10)
line_1.set_color("C2")
ax.set_ylabel("Momentum / kgm/s")

ax.legend(handles=[line_0, line_1], labels=["Position", "Momentum"])
ax.set_title("Closed Dynamics of Harmonic Oscillator")
